# Modelos de la batería
## Descarga del dataset

In [1]:
%pip install adtof@git+https://github.com/MZehren/ADTOF
%pip install yt-dlp

Defaulting to user installation because normal site-packages is not writeable
  Cloning https://github.com/MZehren/ADTOF to c:\users\hp\appdata\local\temp\pip-install-p2cy77nm\adtof_6a34f63b532640c2b07be55d2e305ee9
  Resolved https://github.com/MZehren/ADTOF to commit b3968fb332f69b65ee07c089fc62f436503755db
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  Running command git clone --filter=blob:none --quiet https://github.com/MZehren/ADTOF 'C:\Users\HP\AppData\Local\Temp\pip-install-p2cy77nm\adtof_6a34f63b532640c2b07be55d2e305ee9'
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> [8 lines of output]
      Traceback (most recent call last):
        File "<string>", line 2, in <module>
        File "<pip-setuptools-caller>", line 34, in <module>
        File "C:\Users\HP\AppData\Local\Temp\pip-install-p2cy77nm\adtof_6a34f63b532640c2b07be55d2e305ee9\setup.py", line 4, in <module>
          long_description = fh.read()
        File "c:\Program Files\Python310\lib\encodings\cp1252.py", line 23, in decode
          return codecs.charmap_decode(input,self.errors,decoding_table)[0]
      UnicodeDecodeError: 'charmap' codec can't decode byte 0x9d in position 198: character maps to <undefined>
      [end of output]
  
  note: This error originates from a subprocess, and is lik

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Baseline

In [ ]:
import math, torch, torch.nn as nn, torch.nn.functional as F

def pool_same(x, k=3):
    """MaxPool (1,k) sobre frecuencia, con padding 'same' al estilo Keras."""
    f = x.shape[-1]
    total = max(math.ceil(f / k) * k - f, 0)
    left = total // 2
    x = F.pad(x, (left, total - left), value=float("-inf"))
    return F.max_pool2d(x, (1, k))

class FrameRNN(nn.Module):
    def __init__(self, filters=(32, 64), gru_units=60, n_classes=5):
        super().__init__()
        self.cnn_blocks = nn.ModuleList()
        c_in = 1
        for f in filters:
            self.cnn_blocks.append(nn.Sequential(
                nn.Conv2d(c_in, f, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(f),
                nn.Conv2d(f, f, 3, padding=1),    nn.ReLU(), nn.BatchNorm2d(f)))
            c_in = f
        self.gru_layers = nn.ModuleList([
            nn.GRU(640, gru_units, batch_first=True, bidirectional=True),
            nn.GRU(2 * gru_units, gru_units, batch_first=True, bidirectional=True),
            nn.GRU(2 * gru_units, gru_units, batch_first=True, bidirectional=True)])
        self.output_layer = nn.Linear(2 * gru_units, n_classes)

    def forward(self, x):                        # x: (B, T, 84)
        x = x.unsqueeze(1)                       # (B, 1, T, 84)
        for blk in self.cnn_blocks:
            x = pool_same(blk(x))                # frecuencia: 84 -> 28 -> 10
        x = x.permute(0, 2, 3, 1).reshape(x.shape[0], x.shape[2], -1)  # (B, T, 640)
        for g in self.gru_layers:
            x, _ = g(x)
        return torch.sigmoid(self.output_layer(x))   # (B, T, 5)

ckpt = torch.load("adtof_frame_rnn_pytorch_weights.pth", map_location="cpu")
model = FrameRNN()
model.load_state_dict(ckpt["model_weights"])
model = model.float().eval()

NoSuchFile: [ONNXRuntimeError] : 3 : NO_SUCHFILE : Load model from adtof_frame_rnn.onnx failed:Load model adtof_frame_rnn.onnx failed. File doesn't exist